In [1]:
import time
import csv
from urllib.parse import urlparse
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By

START_URL = "https://www.hipflat.co.th/thailand-projects/condo/bangkok-bm"
OUTPUT_FILE = "hipflat_listing_urls.csv"
TIMEOUT = 40
MAX_PAGES = 20


def wait_ready(driver):
    t0 = time.time()
    while time.time() - t0 < TIMEOUT:
        if driver.execute_script("return document.readyState") == "complete":
            return True
        time.sleep(0.2)
    return False


def check_server_error(driver):
    page_source = driver.page_source.lower()
    if (
        "server error in '/' application" in page_source
        or "timeout expired" in page_source
        or "502 bad gateway" in page_source
        or "504 gateway time-out" in page_source
    ):
        return True
    return False


def extract_links(driver, base):
    js = """
    return [...new Set(
        Array.from(document.querySelectorAll(".project-snippet > a"))
        .map(a => a.getAttribute('href'))
        .filter(Boolean)
    )];
    """
    hrefs = driver.execute_script(js)

    out = []
    for h in hrefs:
        out.append(base + h if h.startswith("/") else h)

    return list(dict.fromkeys(out))


def scroll(driver):
    last_h = 0
    for _ in range(8):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.0)
        h = driver.execute_script("return document.body.scrollHeight")
        if h == last_h:
            break
        last_h = h


def click_next(driver):
    try:
        next_btn = driver.find_element(
            By.XPATH,
            "//li[contains(@class, 'page-arrow') and text()='>']"
        )

        driver.execute_script(
            "arguments[0].scrollIntoView({block:'center'});",
            next_btn
        )
        time.sleep(1)

        driver.execute_script("arguments[0].click();", next_btn)
        return True
    except Exception:
        return False


def main():
    opt = uc.ChromeOptions()
    opt.add_argument("--no-sandbox")
    opt.add_argument("--disable-dev-shm-usage")
    opt.add_argument("--disable-gpu")
    opt.add_argument("--disable-notifications")
    opt.add_argument("--disable-blink-features=AutomationControlled")

    driver = uc.Chrome(options=opt, version_main=145)
    base = f"{urlparse(START_URL).scheme}://{urlparse(START_URL).netloc}"

    print(f"Loading initial URL: {START_URL}")
    driver.get(START_URL)
    wait_ready(driver)
    time.sleep(3)

    page = 1
    urls = set()
    retries = 0
    max_retries = 3

    while page <= MAX_PAGES:
        if check_server_error(driver):
            print(f"[!] Server error on page {page}. Retry {retries + 1}/{max_retries}")
            time.sleep(10)
            driver.refresh()
            wait_ready(driver)
            time.sleep(5)
            retries += 1
            if retries > max_retries:
                print("[!] Max retries reached. Stopping.")
                break
            continue

        retries = 0

        print(f"[Page {page}/{MAX_PAGES}] Collecting links...")
        scroll(driver)
        found = extract_links(driver, base)

        if found:
            urls.update(found)
            print(f"  -> Found {len(found)} listings (total: {len(urls)})")
        else:
            print("  -> Found 0 listings")

        if page >= MAX_PAGES:
            print(f"  -> Reached max pages ({MAX_PAGES})")
            break

        if not click_next(driver):
            print("  -> No next button found. Stopping.")
            break

        print(f"[Page {page}] Next clicked")
        time.sleep(4)
        wait_ready(driver)
        page += 1

    driver.quit()

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["ListingURL"])
        for u in sorted(urls):
            writer.writerow([u])

    print(f"\nDone. Total {len(urls)} listings saved to {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Loading initial URL: https://www.hipflat.co.th/thailand-projects/condo/bangkok-bm
[Page 1/20] Collecting links...
  -> Found 30 listings (total: 30)
[Page 1] Next clicked
[Page 2/20] Collecting links...
  -> Found 30 listings (total: 60)
[Page 2] Next clicked
[Page 3/20] Collecting links...
  -> Found 30 listings (total: 90)
[Page 3] Next clicked
[Page 4/20] Collecting links...
  -> Found 30 listings (total: 120)
[Page 4] Next clicked
[Page 5/20] Collecting links...
  -> Found 30 listings (total: 150)
[Page 5] Next clicked
[Page 6/20] Collecting links...
  -> Found 30 listings (total: 180)
[Page 6] Next clicked
[Page 7/20] Collecting links...
  -> Found 30 listings (total: 210)
[Page 7] Next clicked
[Page 8/20] Collecting links...
  -> Found 30 listings (total: 240)
[Page 8] Next clicked
[Page 9/20] Collecting links...
  -> Found 30 listings (total: 270)
[Page 9] Next clicked
[Page 10/20] Collecting links...
  -> Found 30 listings (total: 300)
[Page 10] Next clicked
[Page 11/20] Collec

In [3]:
import csv
import logging
from pathlib import Path
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# --- Configuration ---
BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/hipflat')
INPUT_CSV = BASE_DIR / 'hipflat_listing_urls.csv'
OUTPUT_CSV = BASE_DIR / 'hipflat_full_details.csv'

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


def setup_optimized_driver() -> uc.Chrome:
    """Configures Chrome for maximum text-scraping performance."""
    options = uc.ChromeOptions()
    
    # 1. Page Load Strategy: 'eager' (Waits only for DOM, skips full resource loading)
    options.page_load_strategy = 'eager'
    
    # 2. Block unnecessary resources (Images, CSS, Fonts) to save bandwidth and CPU
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.default_content_setting_values.media_stream": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
    }
    options.add_experimental_option("prefs", prefs)
    
    # 3. Standard performance arguments
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")
    options.add_argument("--blink-settings=imagesEnabled=false")
    
    return uc.Chrome(options=options, version_main=145)


def extract_content(driver: uc.Chrome) -> str:
    """Extracts property details based on predefined logic."""
    full_content = []
    
    # --- Section 1: Title, Location & Characteristics ---
    try:
        title_text = driver.find_element(By.CSS_SELECTOR, '.main-header .title h1').text.strip()
        if title_text:
            full_content.extend(["--- Listing Title ---", title_text])

        loc_text = driver.find_element(By.CSS_SELECTOR, '.main-header .title .location').text.strip()
        if loc_text:
            full_content.append(f"Location: {loc_text}")
            
        characteristics = driver.find_elements(By.CSS_SELECTOR, '.main-header .characteristics > div')
        if characteristics:
            char_list = [
                f"{c.find_element(By.CLASS_NAME, 'text').text.strip()}: {c.find_element(By.CLASS_NAME, 'data').text.strip()}"
                for c in characteristics
            ]
            full_content.append(" | ".join(char_list))
    except NoSuchElementException:
        pass

    # --- Section 2: Price Information ---
    try:
        price_blocks = driver.find_elements(By.CSS_SELECTOR, '.price .price__operation-type')
        if price_blocks:
            full_content.append("\n--- Price Information ---")
            for block in price_blocks:
                label = block.find_element(By.CLASS_NAME, 'label').text.replace('\n', ' ').strip()
                value = block.find_element(By.CLASS_NAME, 'value').text.strip()
                full_content.append(f"{label}: {value}")
    except NoSuchElementException:
        pass

    # --- Section 3 & 4: Property Details & Description ---
    try:
        # Click "View More" fast using JS if it exists
        try:
            view_more_label = driver.find_element(By.CSS_SELECTOR, 'label.label-viewmore')
            driver.execute_script("arguments[0].click();", view_more_label)
        except NoSuchElementException:
            pass 

        # Extract description directly
        desc_text = driver.find_element(By.ID, 'description').text.strip()
        if desc_text:
            full_content.append("\n--- Property Details ---")
            full_content.append(desc_text.replace("ดูเพิ่มเติม", "").strip())
            
    except NoSuchElementException:
        pass

    return "\n".join(full_content)


def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    # Fast file reading using list comprehension
    if not INPUT_CSV.exists():
        logging.warning(f"File not found: {INPUT_CSV}")
        return

    with INPUT_CSV.open('r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader, None) # Skip Header
        urls = [row[0] for row in reader if row]

    if not urls:
        logging.info("No URLs found to scrape.")
        return

    logging.info(f"Starting to scrape {len(urls)} items...")
    driver = setup_optimized_driver()
    wait = WebDriverWait(driver, 5) # Explicit wait up to 5 seconds

    with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Post_URL', 'Full_Content'])

        for i, url in enumerate(urls, 1):
            try:
                driver.get(url)
                
                # Replace time.sleep(3) with Explicit Wait for the main container
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '.main-header .title h1')))
                
                content = extract_content(driver)
                writer.writerow([url, content])
                
                if i % 10 == 0:
                    logging.info(f"Progress: [{i}/{len(urls)}] scraped.")

            except TimeoutException:
                logging.error(f"[{i}/{len(urls)}] Timeout loading: {url}")
            except Exception as e:
                logging.error(f"[{i}/{len(urls)}] Error scraping {url}: {e}")

    driver.quit()
    logging.info(f"Scraping completed. Saved to: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

2026-03-29 14:11:51,899 - INFO - Starting to scrape 600 items...
2026-03-29 14:12:00,709 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 14:12:12,220 - INFO - Progress: [10/600] scraped.
2026-03-29 14:12:22,277 - INFO - Progress: [20/600] scraped.
2026-03-29 14:12:34,618 - INFO - Progress: [30/600] scraped.
2026-03-29 14:12:44,097 - INFO - Progress: [40/600] scraped.
2026-03-29 14:12:54,195 - INFO - Progress: [50/600] scraped.
2026-03-29 14:13:04,913 - INFO - Progress: [60/600] scraped.
2026-03-29 14:13:15,541 - INFO - Progress: [70/600] scraped.
2026-03-29 14:13:27,425 - INFO - Progress: [80/600] scraped.
2026-03-29 14:13:38,606 - INFO - Progress: [90/600] scraped.
2026-03-29 14:13:51,315 - INFO - Progress: [100/600] scraped.
2026-03-29 14:14:11,147 - INFO - Progress: [110/600] scraped.
2026-03-29 14:14:27,454 - INFO - Progress: [120/600] scraped.
2026-03-29 14:14:56,187 - INFO - Progress: [130/600] scraped.
2026

In [ ]:
import csv
import logging
from pathlib import Path
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# --- Configuration ---
BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/hipflat')
INPUT_CSV = BASE_DIR / 'hipflat_listing_urls.csv'
OUTPUT_CSV = BASE_DIR / 'hipflat_full_details.csv'

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


def setup_optimized_driver() -> uc.Chrome:
    """Configures Chrome for maximum text-scraping performance."""
    options = uc.ChromeOptions()
    
    # 1. Page Load Strategy: 'eager' (Waits only for DOM, skips full resource loading)
    options.page_load_strategy = 'eager'
    
    # 2. Block unnecessary resources (Images, CSS, Fonts) to save bandwidth and CPU
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.default_content_setting_values.media_stream": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
    }
    options.add_experimental_option("prefs", prefs)
    
    # 3. Standard performance arguments
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")
    options.add_argument("--blink-settings=imagesEnabled=false")
    
    return uc.Chrome(options=options, version_main=145)


def extract_content(driver: uc.Chrome) -> str:
    """Extracts property details based on predefined logic."""
    full_content = []
    
    # --- Section 1: Title, Location & Characteristics ---
    try:
        title_text = driver.find_element(By.CSS_SELECTOR, '.main-header .title h1').text.strip()
        if title_text:
            full_content.extend(["--- Listing Title ---", title_text])

        loc_text = driver.find_element(By.CSS_SELECTOR, '.main-header .title .location').text.strip()
        if loc_text:
            full_content.append(f"Location: {loc_text}")
            
        characteristics = driver.find_elements(By.CSS_SELECTOR, '.main-header .characteristics > div')
        if characteristics:
            char_list = [
                f"{c.find_element(By.CLASS_NAME, 'text').text.strip()}: {c.find_element(By.CLASS_NAME, 'data').text.strip()}"
                for c in characteristics
            ]
            full_content.append(" | ".join(char_list))
    except NoSuchElementException:
        pass

    # --- Section 2: Price Information ---
    try:
        price_blocks = driver.find_elements(By.CSS_SELECTOR, '.price .price__operation-type')
        if price_blocks:
            full_content.append("\n--- Price Information ---")
            for block in price_blocks:
                label = block.find_element(By.CLASS_NAME, 'label').text.replace('\n', ' ').strip()
                value = block.find_element(By.CLASS_NAME, 'value').text.strip()
                full_content.append(f"{label}: {value}")
    except NoSuchElementException:
        pass

    # --- Section 3 & 4: Property Details & Description ---
    try:
        # Click "View More" fast using JS if it exists
        try:
            view_more_label = driver.find_element(By.CSS_SELECTOR, 'label.label-viewmore')
            driver.execute_script("arguments[0].click();", view_more_label)
        except NoSuchElementException:
            pass 

        # Extract description directly
        desc_text = driver.find_element(By.ID, 'description').text.strip()
        if desc_text:
            full_content.append("\n--- Property Details ---")
            full_content.append(desc_text.replace("ดูเพิ่มเติม", "").strip())
            
    except NoSuchElementException:
        pass

    return "\n".join(full_content)


def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    # Fast file reading using list comprehension
    if not INPUT_CSV.exists():
        logging.warning(f"File not found: {INPUT_CSV}")
        return

    with INPUT_CSV.open('r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader, None) # Skip Header
        urls = [row[0] for row in reader if row]

    if not urls:
        logging.info("No URLs found to scrape.")
        return

    logging.info(f"Starting to scrape {len(urls)} items...")
    driver = setup_optimized_driver()
    wait = WebDriverWait(driver, 5) # Explicit wait up to 5 seconds

    with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Post_URL', 'Full_Content'])

        for i, url in enumerate(urls, 1):
            try:
                driver.get(url)
                
                # Replace time.sleep(3) with Explicit Wait for the main container
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '.main-header .title h1')))
                
                content = extract_content(driver)
                writer.writerow([url, content])
                
                if i % 10 == 0:
                    logging.info(f"Progress: [{i}/{len(urls)}] scraped.")

            except TimeoutException:
                logging.error(f"[{i}/{len(urls)}] Timeout loading: {url}")
            except Exception as e:
                logging.error(f"[{i}/{len(urls)}] Error scraping {url}: {e}")

    driver.quit()
    logging.info(f"Scraping completed. Saved to: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

2026-03-29 14:41:12,713 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 14:41:15,354 - INFO - Waiting 40 seconds for manual login...
2026-03-29 14:42:37,588 - INFO - Progress: [10/600] scraped.
2026-03-29 14:42:58,106 - INFO - Progress: [20/600] scraped.
2026-03-29 14:43:24,741 - INFO - Progress: [30/600] scraped.
2026-03-29 14:44:00,334 - INFO - Progress: [40/600] scraped.
2026-03-29 14:44:29,378 - INFO - Progress: [50/600] scraped.
2026-03-29 14:44:57,467 - INFO - Progress: [60/600] scraped.
2026-03-29 14:45:23,390 - INFO - Progress: [70/600] scraped.
2026-03-29 14:45:47,005 - INFO - Progress: [80/600] scraped.
2026-03-29 14:46:10,865 - INFO - Progress: [90/600] scraped.
2026-03-29 14:46:35,287 - INFO - Progress: [100/600] scraped.
2026-03-29 14:47:00,020 - INFO - Progress: [110/600] scraped.
2026-03-29 14:47:25,834 - INFO - Progress: [120/600] scraped.
2026-03-29 14:47:52,451 - INFO - Progress: [130/600] scrape